# Извлечение документа через `extract_document()`

Notebook читает сырой пользовательский текст из `raw_prompt.txt`, вызывает проектную функцию `extract_document()` и сохраняет валидированный документ в `extracted_document.json`. Extraction-промпт загружается самой функцией из каталога `prompts/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден корень проекта с pyproject.toml")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
MY_TESTS_DIR = PROJECT_ROOT / "my_tests"

from report_system.config import Settings
from report_system.domain import DocumentType
from report_system.llm import OllamaProvider, extract_document

RAW_INPUT_PATH = MY_TESTS_DIR / "raw_prompt.txt"
OUTPUT_PATH = MY_TESTS_DIR / "extracted_document.json"
DOCUMENT_TYPE = DocumentType.MANUFACTURING_ACT
MODEL = "qwen3.5-unsloth-q6:latest"

settings = Settings()
provider = OllamaProvider(
    base_url=settings.ollama_url,
    model=MODEL,
    timeout=1800.0,
)

In [2]:
raw_inputp = RAW_INPUT_PATH.read_text(encoding="utf-8").strip()
if not raw_inputp:
    raise ValueError(f"Файл {RAW_INPUT_PATH} пуст")

print(f"Прочитано символов: {len(raw_inputp)}")

Прочитано символов: 692


In [3]:
document = extract_document(
    provider=provider,
    prompts_dir=settings.prompts_dir,
    document_type=DOCUMENT_TYPE,
    raw_input=raw_inputp,
)

OUTPUT_PATH.write_text(document.model_dump_json(indent=2), encoding="utf-8")

10493

In [35]:
print(f"Тип: {document.document_type.value}")
print(f"Статус: {document.status.value}")
print(f"Разделов: {len(document.sections)}")
print(f"Результат: {OUTPUT_PATH.resolve()}")
print("\nИзвлечённый документ:\n")
print(document.model_dump_json(indent=2))

Тип: manufacturing_act
Статус: extracted
Разделов: 5
Результат: /Users/user/Desktop/Code/ReportBot/my_tests/extracted_document.json

Извлечённый документ:

{
  "id": "bb19d7cd-12ce-4dda-bfe9-61e7447fb79d",
  "document_type": "manufacturing_act",
  "title": null,
  "metadata": {
    "document_type": "manufacturing_act",
    "source_type": "user_input"
  },
  "sections": [
    {
      "name": "raw_materials",
      "description": "Исходные материалы и их количества",
      "records": [
        {
          "type": "powder",
          "name": "БА-17",
          "description": null,
          "parameters": [
            {
              "key": null,
              "name": "mass",
              "value": 300,
              "unit": "г",
              "value_type": "scalar",
              "source": null
            }
          ]
        },
        {
          "type": "solvent_mixture",
          "name": "MEK/EtOH",
          "description": "70/30",
          "parameters": [
            {
        